In [21]:
import subprocess
import os
import argparse

In [22]:
# parser = argparse.ArgumentParser(
#     description="proGAT ONT prokaryotic genome assembly and QC pipeline"
# )
# parser.add_argument(
#     "-i", "--input",
#     required=True,
#     help="input_file，default: .fastq/.fastq.gz 文件",
# )
# parser.add_argument(
#     "-o", "--output",
#     default="./proGAT_output",
#     help="output_folder， default: ./proGAT_output",
# )

# args = parser.parse_args()

# input_dir = args.input
# output_dir = args.output

In [ ]:
input_dir = os.path.join("/home/led/temp/proGAT/temp/datas", "SRR39189772_fastq.gz")
output_dir = os.path.join("/home/led/temp/proGAT/temp/output")
database_path = "/home/led/database/"

# 1.seqkit统计

In [24]:
seqkit_dir = os.path.join(output_dir, "seqkit")
os.makedirs(seqkit_dir, exist_ok=True)

docker_image_seqkit = "staphb/seqkit:2.13.0"

cmd = [
    "docker", "run", "--rm",
    "-v", f"{input_dir}:/input.fastq.gz:ro",
    "-v", f"{seqkit_dir}:/output",
    docker_image_seqkit,
    "seqkit", "stats",
    "/input.fastq.gz",
    "-aT",
    "-o", "/output/seqkit_raw.txt"
]

subprocess.run(cmd, check=True)

CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/datas/SRR39189772_fastq.gz:/input.fastq.gz:ro', '-v', '/home/led/temp/proGAT/temp/output/seqkit:/output', 'staphb/seqkit:2.13.0', 'seqkit', 'stats', '/input.fastq.gz', '-aT', '-o', '/output/seqkit_raw.txt'], returncode=0)

# 2.低质基因过滤

In [25]:
fastplong_dir = os.path.join(output_dir, "fastplong")
os.makedirs(fastplong_dir, exist_ok=True)

docker_image_fastplong = "staphb/fastplong:0.7.0"

cmd = [
    "docker", "run", "--rm",
    "-v", f"{input_dir}:/input.fastq.gz:ro",
    "-v", f"{fastplong_dir}:/output_dir",
    docker_image_fastplong,
    "fastplong",
    "-i", "/input.fastq.gz",
    "-o", "/output_dir/filtered.fastq.gz",
    "-h", "/output_dir/fastplong.html",
    "-j", "/output_dir/fastplong.json"
]

subprocess.run(cmd, check=True)

Trying to detect adapter sequence at read start
Detected: GTTTTCGCATTTATCGTGAAACGCTTTCGCGTTTT
Trying to detect adapter sequence at read end
Not detected

Before filtering:
total reads: 357377
total bases: 1929681344
Q20 bases: 1173395088(60.8077%)
Q30 bases: 300374656(15.566%)

After filtering:
total reads: 316317
total bases: 1694277850
Q20 bases: 1079750598(63.7293%)
Q30 bases: 281627956(16.6223%)

Filtering result:
reads passed filter: 316317
reads failed due to low quality: 41311
reads failed due to too many N: 0
reads failed due to too short: 25
reads with adapter trimmed: 346637
bases trimmed due to adapters: 36727680
reads with internal adapters/chimeric signals: 825
bases removed by chimera splitting: 46000
segments produced by chimera splitting: 1101

JSON report: /output_dir/fastplong.json
HTML report: /output_dir/fastplong.html

fastplong -i /input.fastq.gz -o /output_dir/filtered.fastq.gz -h /output_dir/fastplong.html -j /output_dir/fastplong.json 
fastplong v0.7.0, time us

CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/datas/SRR39189772_fastq.gz:/input.fastq.gz:ro', '-v', '/home/led/temp/proGAT/temp/output/fastplong:/output_dir', 'staphb/fastplong:0.7.0', 'fastplong', '-i', '/input.fastq.gz', '-o', '/output_dir/filtered.fastq.gz', '-h', '/output_dir/fastplong.html', '-j', '/output_dir/fastplong.json'], returncode=0)

In [26]:
seqkit_dir = os.path.join(output_dir, "seqkit")
os.makedirs(seqkit_dir, exist_ok=True)

docker_image_seqkit = "staphb/seqkit:2.13.0"

input_file = os.path.join(fastplong_dir, "filtered.fastq.gz")

cmd = [
    "docker", "run", "--rm",
    "-v", f"{input_file}:/input.fastq.gz:ro",
    "-v", f"{seqkit_dir}:/output",
    docker_image_seqkit,
    "seqkit", "stats",
    "/input.fastq.gz",
    "-aT",
    "-o", "/output/seqkit_clean.txt"
]

subprocess.run(cmd, check=True)

CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/fastplong/filtered.fastq.gz:/input.fastq.gz:ro', '-v', '/home/led/temp/proGAT/temp/output/seqkit:/output', 'staphb/seqkit:2.13.0', 'seqkit', 'stats', '/input.fastq.gz', '-aT', '-o', '/output/seqkit_clean.txt'], returncode=0)

# 3. 基因组大小评估

In [29]:
lrge_dir = os.path.join(output_dir, "lrge")
os.makedirs(lrge_dir, exist_ok=True)

docker_image_lrge = "staphb/lrge:1.0.0"

input_file = os.path.join(fastplong_dir, "filtered.fastq.gz")

output_file = "est_size.txt"

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fastq.gz:ro", 
    "-v", f"{lrge_dir}:/output", 
    docker_image_lrge, 
    "lrge", 
    "-P", "ont", 
    "-t", "128", 
    "-o", f"/output/{output_file}", 
    "/input.fastq.gz"
]

subprocess.run(cmd, check=True)

[2026-09-29T09:57:24Z INFO  lrge] Running two-set strategy with 10000 target reads and 5000 query reads
[2026-09-29T09:58:00Z WARN  liblrge::twoset] Depth skew detected (99.9th percentile minimizer count is 21.00x the median; sampled reads: 3176); depth normalization retained 286148 of 316317 reads
[2026-09-29T09:58:02Z INFO  liblrge::twoset] 72 (1.44%) query read(s) did not overlap any target reads
[2026-09-29T09:58:02Z INFO  lrge] Estimated genome size: 5.67 Mbp (95% CI: 4.42 Mbp - 6.27 Mbp)
[2026-09-29T09:58:02Z INFO  lrge] Done!


CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/fastplong/filtered.fastq.gz:/input.fastq.gz:ro', '-v', '/home/led/temp/proGAT/temp/output/lrge:/output', 'staphb/lrge:1.0.0', 'lrge', '-P', 'ont', '-t', '128', '-o', '/output/est_size.txt', '/input.fastq.gz'], returncode=0)

# 4. 基因组拼接-flye

In [30]:
flye_dir = os.path.join(output_dir, "flye")
os.makedirs(flye_dir, exist_ok=True)

docker_image_flye = "staphb/flye:2.9.6"

input_file = os.path.join(fastplong_dir, "filtered.fastq.gz")

lrge_result = os.path.join(
    lrge_dir,
    "est_size.txt"
)

with open(lrge_result) as f:
    genome_size = f.read().strip()

print(f"Estimated genome size: {genome_size}")

cmd = [
    "docker", "run", "--rm",
    "-v", f"{input_file}:/input.fastq.gz:ro",
    "-v", f"{flye_dir}:/output",
    docker_image_flye,
    "flye",
    "--nano-hq", "/input.fastq.gz",
    "--genome-size", genome_size,
    "--out-dir", "/output",
    "--threads", "128"
]

subprocess.run(cmd, check=True)

Estimated genome size: 5668948


[2026-09-29 09:58:03] INFO: Starting Flye 2.9.6-b1802
[2026-09-29 09:58:03] INFO: >>>STAGE: configure
[2026-09-29 09:58:03] INFO: Configuring run
[2026-09-29 09:58:29] INFO: Total read length: 1694277850
[2026-09-29 09:58:29] INFO: Input genome size: 5668948
[2026-09-29 09:58:29] INFO: Estimated coverage: 298
[2026-09-29 09:58:29] INFO: Reads N50/N90: 10345 / 2738
[2026-09-29 09:58:29] INFO: Minimum overlap set to 3000
[2026-09-29 09:58:29] INFO: >>>STAGE: assembly
[2026-09-29 09:58:29] INFO: Assembling disjointigs
[2026-09-29 09:58:29] INFO: Reading sequences
[2026-09-29 09:58:51] INFO: Building minimizer index
[2026-09-29 09:58:51] INFO: Pre-calculating index storage
0% 10% 20% 30% 40% 50% 60% 70% 80% 90% 100% 
[2026-09-29 09:59:04] INFO: Filling index
0% 10% 20% 30% 40% 50% 60% 70% 80% 90% 100% 
[2026-09-29 09:59:18] INFO: Extending reads
[2026-09-29 10:03:25] INFO: Overlap-based coverage: 242
[2026-09-29 10:03:25] INFO: Median overlap divergence: 0.0676668
0% 90% 100% 
[2026-09-29 

CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/fastplong/filtered.fastq.gz:/input.fastq.gz:ro', '-v', '/home/led/temp/proGAT/temp/output/flye:/output', 'staphb/flye:2.9.6', 'flye', '--nano-hq', '/input.fastq.gz', '--genome-size', '5668948', '--out-dir', '/output', '--threads', '128'], returncode=0)

# 4. 基因组拼接-hifiasm

In [33]:
hifiasm_dir = os.path.join(output_dir, "hifiasm")
os.makedirs(hifiasm_dir, exist_ok=True)

docker_image_hifiasm = "staphb/hifiasm:0.25.0"

input_file = os.path.join(fastplong_dir, "filtered.fastq.gz")

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fastq.gz:ro",
    "-v", f"{hifiasm_dir}:/output", 
    docker_image_hifiasm, 
    "hifiasm", 
    "--ont", 
    "-t", "128", 
    "-o", "/output/ONT.asm",
    "/input.fastq.gz"
]

subprocess.run(cmd, check=True)

[M::ha_analyze_count] lowest: count[36] = 10143
[M::ha_analyze_count] highest: count[109] = 50411
[M::ha_hist_line]     2: ****************************************************************************************************> 37008164
[M::ha_hist_line]     3: ****************************************************************************************************> 12959125
[M::ha_hist_line]     4: ****************************************************************************************************> 6226815
[M::ha_hist_line]     5: ****************************************************************************************************> 3469092
[M::ha_hist_line]     6: ****************************************************************************************************> 2120531
[M::ha_hist_line]     7: ****************************************************************************************************> 1377711
[M::ha_hist_line]     8: *****************************************************************

CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/fastplong/filtered.fastq.gz:/input.fastq.gz:ro', '-v', '/home/led/temp/proGAT/temp/output/hifiasm:/output', 'staphb/hifiasm:0.25.0', 'hifiasm', '--ont', '-t', '128', '-o', '/output/ONT.asm', '/input.fastq.gz'], returncode=0)

In [38]:
gfa_file = os.path.join(
    hifiasm_dir,
    "ONT.asm.bp.p_ctg.gfa"
)

assembly_file = os.path.join(
    hifiasm_dir,
    "assembly.fasta"
)

if not os.path.isfile(gfa_file):
    raise FileNotFoundError(
        f"Hifiasm output not found: {gfa_file}"
    )

with open(gfa_file, "r") as infile, \
     open(assembly_file, "w") as outfile:

    for line in infile:
        if line.startswith("S\t"):
            fields = line.rstrip().split("\t")

            seq_id = fields[1]
            sequence = fields[2]

            outfile.write(
                f">{seq_id}\n{sequence}\n"
            )

# 5. 拼接结果评估

## 5.1 QUAST

In [ ]:
quast_dir = os.path.join(output_dir, "quast")
os.makedirs(quast_dir, exist_ok=True)
docker_image_quast = "staphb/quast:5.3.0"

In [ ]:
# quast_flye
temp_dir= os.path.join(quast_dir, "flye")
os.makedirs(temp_dir, exist_ok=True)

input_file = os.path.join(flye_dir, "assembly.fasta")

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta", 
    "-v", f"{temp_dir}:/output", 
    docker_image_quast, 
    "quast.py", 
    "/input.fasta", 
    "-o", "/output", 
    "-t", "128"
]

subprocess.run(cmd, check=True)

/quast-5.3.0/quast.py /input.fasta -o /output -t 128

Version: 5.3.0

System information:
  OS: Linux-7.0.0-30-generic-x86_64-with-glibc2.35 (linux_64)
  Python version: 3.10.12
  CPUs number: 192

Started: 2026-09-29 21:58:40

Logging to /output/quast.log
NOTICE: Output directory already exists and looks like a QUAST output dir. Existing results can be reused (e.g. previously generated alignments)!

CWD: /data
Main parameters: 
  MODE: default, threads: 128, min contig length: 500, min alignment length: 65, min alignment IDY: 95.0, \
  ambiguity: one, min local misassembly length: 200, min extensive misassembly length: 1000

Contigs:
  Pre-processing...
  /input.fasta ==> input

2026-09-29 21:58:41
Running Basic statistics processor...
  Contig files: 
    input
  Calculating N50 and L50...
    input, N50 = 5399658, L50 = 1, auN = 5076822.2, Total length = 5754487, GC % = 57.07, # N's per 100 kbp =  0.00
  Drawing Nx plot...
    saved to /output/basic_stats/Nx_plot.pdf
  Drawing cumul

CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/flye/assembly.fasta:/input.fasta', '-v', '/home/led/temp/proGAT/temp/output/quast/flye:/output', 'staphb/quast:5.3.0', 'quast.py', '/input.fasta', '-o', '/output', '-t', '128'], returncode=0)

In [ ]:
# quast_hifiasm
temp_dir= os.path.join(quast_dir, "hifiasm")
os.makedirs(temp_dir, exist_ok=True)

input_file = os.path.join(hifiasm_dir, "assembly.fasta")

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta", 
    "-v", f"{temp_dir}:/output", 
    docker_image_quast, 
    "quast.py", 
    "/input.fasta", 
    "-o", "/output", 
    "-t", "128"
]

subprocess.run(cmd, check=True)

/quast-5.3.0/quast.py /input.fasta -o /output -t 128

Version: 5.3.0

System information:
  OS: Linux-7.0.0-30-generic-x86_64-with-glibc2.35 (linux_64)
  Python version: 3.10.12
  CPUs number: 192

Started: 2026-09-29 21:58:44

Logging to /output/quast.log
NOTICE: Output directory already exists and looks like a QUAST output dir. Existing results can be reused (e.g. previously generated alignments)!

CWD: /data
Main parameters: 
  MODE: default, threads: 128, min contig length: 500, min alignment length: 65, min alignment IDY: 95.0, \
  ambiguity: one, min local misassembly length: 200, min extensive misassembly length: 1000

Contigs:
  Pre-processing...
  /input.fasta ==> input

2026-09-29 21:58:45
Running Basic statistics processor...
  Contig files: 
    input
  Calculating N50 and L50...
    input, N50 = 5527980, L50 = 1, auN = 5110640.5, Total length = 5994669, GC % = 56.85, # N's per 100 kbp =  0.00
  Drawing Nx plot...
    saved to /output/basic_stats/Nx_plot.pdf
  Drawing cumul

CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/hifiasm/assembly.fasta:/input.fasta', '-v', '/home/led/temp/proGAT/temp/output/quast/hifiasm:/output', 'staphb/quast:5.3.0', 'quast.py', '/input.fasta', '-o', '/output', '-t', '128'], returncode=0)

## 5.2 BUSCO

In [ ]:
busco_dir = os.path.join(output_dir, "busco")
os.makedirs(busco_dir, exist_ok=True)
docker_image_busco = "staphb/busco:6.0.0"

busco_db = os.path.join(database_path, "busco")

In [105]:
# 可下载模型list
cmd = [
    "docker", "run", "--rm",
    docker_image_busco, 
    "busco", 
    "--list-datasets"
]

subprocess.run(cmd, check=True)

2026-09-30 06:47:14 INFO:	Downloading information on latest versions of BUSCO data...
2026-09-30 06:47:23 INFO:	Downloading file 'https://busco-data.ezlab.org/v5/data/information/lineages_list.2026-05-26.txt.tar.gz'
2026-09-30 06:47:26 INFO:	Decompressing file '/data/busco_downloads/information/lineages_list.2026-05-26.txt.tar.gz'
################################################

Datasets available to be used with BUSCO v6.1.0 and later (numbers in brackets indicate the number of marker gene profiles):

- archaea_odb12.2 [195]
    - euryarchaeota_odb12.2 [282]
        - methanomicrobia_odb12.2 [589]
            - methanosarcinaceae_odb12.2 [971]
            - methanosarcina_odb12.2 [1620]
            - methanomicrobiales_odb12.2 [819]
               - methanomicrobiaceae_odb12.2 [993]
        - halobacteria_odb12.2 [807]
            - halobacteriales_odb12.2 [909]
                - halobacteriaceae_odb12.2 [975]
                - haloarculaceae_odb12.2 [1058]
                    - halo

CompletedProcess(args=['docker', 'run', '--rm', 'staphb/busco:6.0.0', 'busco', '--list-datasets'], returncode=0)

In [ ]:
# 模型下载
# database_path = "/home/led/database/"
# busco_db = os.path.join(database_path, "busco")
# os.makedirs(busco_db, exist_ok=True)

# cmd = [
#     "docker", "run", "--rm", 
#     "-v", f"{busco_db}:/busco_db", 
#     docker_image_busco, 
#     "busco", 
#     "--download", "bacteria_odb12", 
#     "--download_path", "/busco_db/bacteria_odb12"
# ]

# subprocess.run(cmd, check=True)

2026-09-30 02:27:18 INFO:	Downloading information on latest versions of BUSCO data...
2026-09-30 02:27:28 INFO:	Downloading file 'https://busco-data.ezlab.org/v5/data/lineages/bacteria_odb12.2026-05-22.tar.gz'
2026-09-30 02:46:20 INFO:	Decompressing file '/busco_db/bacteria_odb12/lineages/bacteria_odb12.tar.gz'


CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/database/busco:/busco_db', 'staphb/busco:6.0.0', 'busco', '--download', 'bacteria_odb12', '--download_path', '/busco_db/bacteria_odb12'], returncode=0)

In [125]:
busco_db_temp = os.path.join(busco_db, "bacteria_odb12", "lineages")

In [ ]:
# flye
temp_dir= os.path.join(busco_dir, "flye")
os.makedirs(temp_dir, exist_ok=True)

input_file =os.path.join(flye_dir, "assembly.fasta")

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta:ro", 
    "-v", f"{temp_dir}:/output", 
    "-v", f"{busco_db_temp}:/busco_db:ro",
    docker_image_busco, 
    "busco", 
    "-i", "/input.fasta", 
    "-o", "/busco_result", 
    "--out_path", "/output", 
    "-l", "/busco_db/bacteria_odb12", 
    "-m", "genome", 
    "--cpu", "128", 
    "--offline", "-f", 
]

subprocess.run(cmd, check=True)

# plot
cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta:ro", 
    "-v", f"{temp_dir}:/output", 
    "-v", f"{busco_db_temp}:/busco_db:ro",
    docker_image_busco, 
    "busco", 
    "-i", "/input.fasta", 
    "-o", "/busco_result", 
    "--out_path", "/output", 
    "-l", "/busco_db/bacteria_odb12", 
    "-m", "genome", 
    "--cpu", "128", 
    "--offline", "-f", 
    "--plot", "/output/busco_result", 
    "--plot_percentages"
]

subprocess.run(cmd, check=True)

2026-09-30 03:04:12 INFO:	***** Start a BUSCO v6.0.0 analysis, current time: 09/30/2026 03:04:12 *****
2026-09-30 03:04:12 INFO:	Configuring BUSCO with local environment
2026-09-30 03:04:12 INFO:	Running genome mode
2026-09-30 03:04:12 INFO:	'Force' option selected; overwriting previous results directory
2026-09-30 03:04:12 INFO:	Input file is /input.fasta
2026-09-30 03:04:12 INFO:	Using local lineages directory /busco_db/bacteria_odb12
2026-09-30 03:04:12 INFO:	Running BUSCO using lineage dataset bacteria_odb12 (prokaryota, 2026-05-22)
2026-09-30 03:04:12 INFO:	Running 1 job(s) on bbtools, starting at 09/30/2026 03:04:12
2026-09-30 03:04:14 INFO:	[bbtools]	1 of 1 task(s) completed
2026-09-30 03:04:14 INFO:	***** Run Prodigal on input to predict and extract genes *****
2026-09-30 03:04:14 INFO:	Running Prodigal with genetic code 11 in single mode
2026-09-30 03:04:14 INFO:	Running 1 job(s) on prodigal, starting at 09/30/2026 03:04:14
2026-09-30 03:04:22 INFO:	[prodigal]	1 of 1 task(s) c

CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/flye/assembly.fasta:/input.fasta:ro', '-v', '/home/led/temp/proGAT/temp/output/busco/flye:/output', '-v', '/home/led/database/busco/bacteria_odb12/lineages:/busco_db:ro', 'staphb/busco:6.0.0', 'busco', '-i', '/input.fasta', '-o', '/busco_result', '--out_path', '/output', '-l', '/busco_db/bacteria_odb12', '-m', 'genome', '--cpu', '128', '--offline', '-f', '--plot', '/output/busco_result', '--plot_percentages'], returncode=0)

In [77]:
# hifiasm
temp_dir= os.path.join(busco_dir, "hifiasm")
os.makedirs(temp_dir, exist_ok=True)

input_file =os.path.join(hifiasm_dir, "assembly.fasta")

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta:ro", 
    "-v", f"{temp_dir}:/output", 
    "-v", f"{busco_db_temp}:/busco_db:ro",
    docker_image_busco, 
    "busco", 
    "-i", "/input.fasta", 
    "-o", "/busco_result", 
    "--out_path", "/output", 
    "-l", "/busco_db/bacteria_odb12", 
    "-m", "genome", 
    "--cpu", "128", 
    "--offline", "-f", 
]

subprocess.run(cmd, check=True)

# plot
cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta:ro", 
    "-v", f"{temp_dir}:/output", 
    "-v", f"{busco_db_temp}:/busco_db:ro",
    docker_image_busco, 
    "busco", 
    "-i", "/input.fasta", 
    "-o", "/busco_result", 
    "--out_path", "/output", 
    "-l", "/busco_db/bacteria_odb12", 
    "-m", "genome", 
    "--cpu", "128", 
    "--offline", "-f", 
    "--plot", "/output/busco_result", 
    "--plot_percentages"
]

subprocess.run(cmd, check=True)

2026-09-30 03:06:49 INFO:	***** Start a BUSCO v6.0.0 analysis, current time: 09/30/2026 03:06:49 *****
2026-09-30 03:06:49 INFO:	Configuring BUSCO with local environment
2026-09-30 03:06:49 INFO:	Running genome mode
2026-09-30 03:06:49 INFO:	Input file is /input.fasta
2026-09-30 03:06:49 INFO:	Using local lineages directory /busco_db/bacteria_odb12
2026-09-30 03:06:49 INFO:	Running BUSCO using lineage dataset bacteria_odb12 (prokaryota, 2026-05-22)
2026-09-30 03:06:49 INFO:	Running 1 job(s) on bbtools, starting at 09/30/2026 03:06:49
2026-09-30 03:06:50 INFO:	[bbtools]	1 of 1 task(s) completed
2026-09-30 03:06:51 INFO:	***** Run Prodigal on input to predict and extract genes *****
2026-09-30 03:06:51 INFO:	Running Prodigal with genetic code 11 in single mode
2026-09-30 03:06:51 INFO:	Running 1 job(s) on prodigal, starting at 09/30/2026 03:06:51
2026-09-30 03:07:18 INFO:	[prodigal]	1 of 1 task(s) completed
2026-09-30 03:07:18 INFO:	Genetic code 11 selected as optimal
2026-09-30 03:07:18

CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/hifiasm/assembly.fasta:/input.fasta:ro', '-v', '/home/led/temp/proGAT/temp/output/busco/hifiasm:/output', '-v', '/home/led/database/busco/bacteria_odb12/lineages:/busco_db:ro', 'staphb/busco:6.0.0', 'busco', '-i', '/input.fasta', '-o', '/busco_result', '--out_path', '/output', '-l', '/busco_db/bacteria_odb12', '-m', 'genome', '--cpu', '128', '--offline', '-f', '--plot', '/output/busco_result', '--plot_percentages'], returncode=0)

## 5.3 Merqury

In [126]:
merqury_dir = os.path.join(output_dir, "merqury")
os.makedirs(merqury_dir, exist_ok=True)
docker_image_merqury = "danylmb/merqury:1.4.1-build2"

In [127]:
# flye
temp_dir = os.path.join(merqury_dir, "flye")
os.makedirs(temp_dir, exist_ok=True)

input_file_merqury = os.path.join(flye_dir, "assembly.fasta")
input_file_meryl = os.path.join(fastplong_dir, "filtered.fastq.gz")

read_meryl = os.path.join(temp_dir, "meryl_dir")
os.makedirs(read_meryl, exist_ok=True)

cmd = [
    "docker", "run", "--rm", 
    "--memory", "256g",
    "--cpus", "128", 
    "-v", f"{input_file_meryl}:/input.fastq.gz:ro", 
    "-v", f"{read_meryl}:/output",
    docker_image_merqury, 
    "meryl", 
    "count", 
    "k=17",
    "output", "/output/read.meryl",
    "/input.fastq.gz"
]

subprocess.run(cmd, check=True)

cmd = [
    "docker", "run", "--rm",
    "--memory", "256g",
    "--cpus", "128",
    "-v", f"{read_meryl}:/input_meryl:ro",
    "-v", f"{input_file_merqury}:/input.fasta:ro",
    "-v", f"{temp_dir}:/output",
    "-w", "/output",
    docker_image_merqury,
    "merqury.sh",
    "/input_meryl/read.meryl",
    "/input.fasta",
    "merqury_res"
]

subprocess.run(cmd, check=True)



Found 1 command tree.

Counting 4531 (estimated) million canonical 17-mers from 1 input file:
    sequence-file: /input.fastq.gz


SIMPLE MODE
-----------

  17-mers
    -> 17179869184 entries for counts up to 65535.
    -> 256 Gbits memory used

  4751693655 input bases
    -> expected max count of 19006774, needing 10 extra bits.
    -> 160 Gbits memory used

  52 GB memory needed


COMPLEX MODE
------------

prefix     # of   struct   kmers/    segs/      min     data    total
  bits   prefix   memory   prefix   prefix   memory   memory   memory
------  -------  -------  -------  -------  -------  -------  -------
     1     2  P    18 MB  1132 MM  1169 kS  8192  B  9355 MB  9373 MB
     2     4  P    17 MB   566 MM   566 kS    16 kB  9063 MB  9080 MB
     3     8  P    17 MB   283 MM   274 kS    32 kB  8780 MB  8797 MB
     4    16  P    16 MB   141 MM   132 kS    64 kB  8498 MB  8515 MB
     5    32  P    16 MB    70 MM    64 kS   128 kB  8220 MB  8236 MB
     6    64  P    15 MB

read: read.meryl

No haplotype dbs provided.
Running Merqury in non-trio mode...

asm1: input.fasta
out : merqury_res


Get spectra-cn plots and QV stats


CompletedProcess(args=['docker', 'run', '--rm', '--memory', '256g', '--cpus', '128', '-v', '/home/led/temp/proGAT/temp/output/merqury/flye/meryl_dir:/input_meryl:ro', '-v', '/home/led/temp/proGAT/temp/output/flye/assembly.fasta:/input.fasta:ro', '-v', '/home/led/temp/proGAT/temp/output/merqury/flye:/output', '-w', '/output', 'danylmb/merqury:1.4.1-build2', 'merqury.sh', '/input_meryl/read.meryl', '/input.fasta', 'merqury_res'], returncode=0)

In [128]:
# hifiasm
temp_dir = os.path.join(merqury_dir, "hifiasm")
os.makedirs(temp_dir, exist_ok=True)

input_file_merqury = os.path.join(hifiasm_dir, "assembly.fasta")
input_file_meryl = os.path.join(fastplong_dir, "filtered.fastq.gz")

read_meryl = os.path.join(temp_dir, "meryl_dir")
os.makedirs(read_meryl, exist_ok=True)

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file_meryl}:/input.fastq.gz:ro", 
    "-v", f"{read_meryl}:/output",
    "--memory", "256g",
    "--cpus", "128", 
    docker_image_merqury, 
    "meryl", 
    "count", 
    "k=17",
    "output", "/output/read.meryl",
    "/input.fastq.gz"
]

subprocess.run(cmd, check=True)

cmd = [
    "docker", "run", "--rm",
    "--memory", "256g",
    "--cpus", "128",
    "-v", f"{read_meryl}:/input_meryl:ro",
    "-v", f"{input_file_merqury}:/input.fasta:ro",
    "-v", f"{temp_dir}:/output",
    "-w", "/output",
    docker_image_merqury,
    "merqury.sh",
    "/input_meryl/read.meryl",
    "/input.fasta",
    "merqury_res"
]

subprocess.run(cmd, check=True)


Found 1 command tree.

Counting 4531 (estimated) million canonical 17-mers from 1 input file:
    sequence-file: /input.fastq.gz


SIMPLE MODE
-----------

  17-mers
    -> 17179869184 entries for counts up to 65535.
    -> 256 Gbits memory used

  4751693655 input bases
    -> expected max count of 19006774, needing 10 extra bits.
    -> 160 Gbits memory used

  52 GB memory needed


COMPLEX MODE
------------

prefix     # of   struct   kmers/    segs/      min     data    total
  bits   prefix   memory   prefix   prefix   memory   memory   memory
------  -------  -------  -------  -------  -------  -------  -------
     1     2  P    18 MB  1132 MM  1169 kS  8192  B  9355 MB  9373 MB
     2     4  P    17 MB   566 MM   566 kS    16 kB  9063 MB  9080 MB
     3     8  P    17 MB   283 MM   274 kS    32 kB  8780 MB  8797 MB
     4    16  P    16 MB   141 MM   132 kS    64 kB  8498 MB  8515 MB
     5    32  P    16 MB    70 MM    64 kS   128 kB  8220 MB  8236 MB
     6    64  P    15 MB

read: read.meryl

No haplotype dbs provided.
Running Merqury in non-trio mode...

asm1: input.fasta
out : merqury_res


Get spectra-cn plots and QV stats


CompletedProcess(args=['docker', 'run', '--rm', '--memory', '256g', '--cpus', '128', '-v', '/home/led/temp/proGAT/temp/output/merqury/hifiasm/meryl_dir:/input_meryl:ro', '-v', '/home/led/temp/proGAT/temp/output/hifiasm/assembly.fasta:/input.fasta:ro', '-v', '/home/led/temp/proGAT/temp/output/merqury/hifiasm:/output', '-w', '/output', 'danylmb/merqury:1.4.1-build2', 'merqury.sh', '/input_meryl/read.meryl', '/input.fasta', 'merqury_res'], returncode=0)

## 5.4 CheckM2

In [139]:
checkm_dir = os.path.join(output_dir, "checkm2")
os.makedirs(checkm_dir, exist_ok=True)
docker_image_checkm2 = "staphb/checkm2:1.1.0"

In [140]:
checkm_db_file = os.path.join(database_path, "checkm2", "CheckM2_database", "uniref100.KO.1.dmnd")

In [141]:
# flye
temp_dir = os.path.join(checkm_dir, "flye")
os.makedirs(temp_dir, exist_ok=True)

input_file =os.path.join(flye_dir, "assembly.fasta")

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta", 
    "-v", f"{temp_dir}:/output",
    "-v", f"{checkm_db_file}:/checkm_data.dmnd:ro", 
    docker_image_checkm2, 
    "checkm2", "predict", 
    "--threads", "128", 
    "--input", "/input.fasta", 
    "--output-directory", "/output", 
    "--database", "/checkm_data.dmnd"
]

subprocess.run(cmd, check=True)

[09/30/2026 08:25:31 AM] INFO: Running CheckM2 version 1.1.0
[09/30/2026 08:25:31 AM] INFO: Custom database path provided for predict run. Checking database at /checkm_data.dmnd...
[09/30/2026 08:25:35 AM] INFO: Running quality prediction workflow with 128 threads.
[09/30/2026 08:25:35 AM] INFO: Calling genes in 1 bins with 128 threads:


    Finished processing 1 of 1 (100.00%) bins.


[09/30/2026 08:25:56 AM] INFO: Calculating metadata for 1 bins with 128 threads:


    Finished processing 1 of 1 (100.00%) bin metadata.


[09/30/2026 08:26:01 AM] INFO: Annotating input genomes with DIAMOND using 128 threads
[09/30/2026 08:26:26 AM] INFO: Processing DIAMOND output
[09/30/2026 08:26:26 AM] INFO: Predicting completeness and contamination using ML models.
[09/30/2026 08:26:30 AM] INFO: Parsing all results and constructing final output table.
[09/30/2026 08:26:30 AM] INFO: CheckM2 finished successfully.


CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/flye/assembly.fasta:/input.fasta', '-v', '/home/led/temp/proGAT/temp/output/checkm2/flye:/output', '-v', '/home/led/database/checkm2/CheckM2_database/uniref100.KO.1.dmnd:/checkm_data.dmnd:ro', 'staphb/checkm2:1.1.0', 'checkm2', 'predict', '--threads', '128', '--input', '/input.fasta', '--output-directory', '/output', '--database', '/checkm_data.dmnd'], returncode=0)

In [142]:
# hifiasm
temp_dir = os.path.join(checkm_dir, "hifiasm")
os.makedirs(temp_dir, exist_ok=True)

input_file =os.path.join(hifiasm_dir, "assembly.fasta")

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta", 
    "-v", f"{temp_dir}:/output",
    "-v", f"{checkm_db_file}:/checkm_data.dmnd:ro", 
    docker_image_checkm2, 
    "checkm2", "predict", 
    "--threads", "128", 
    "--input", "/input.fasta", 
    "--output-directory", "/output", 
    "--database", "/checkm_data.dmnd"
]

subprocess.run(cmd, check=True)

[09/30/2026 08:26:35 AM] INFO: Running CheckM2 version 1.1.0
[09/30/2026 08:26:35 AM] INFO: Custom database path provided for predict run. Checking database at /checkm_data.dmnd...
[09/30/2026 08:26:39 AM] INFO: Running quality prediction workflow with 128 threads.
[09/30/2026 08:26:39 AM] INFO: Calling genes in 1 bins with 128 threads:


    Finished processing 1 of 1 (100.00%) bins.


[09/30/2026 08:27:02 AM] INFO: Calculating metadata for 1 bins with 128 threads:


    Finished processing 1 of 1 (100.00%) bin metadata.


[09/30/2026 08:27:07 AM] INFO: Annotating input genomes with DIAMOND using 128 threads
[09/30/2026 08:27:33 AM] INFO: Processing DIAMOND output
[09/30/2026 08:27:33 AM] INFO: Predicting completeness and contamination using ML models.
[09/30/2026 08:27:37 AM] INFO: Parsing all results and constructing final output table.
[09/30/2026 08:27:37 AM] INFO: CheckM2 finished successfully.


CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/hifiasm/assembly.fasta:/input.fasta', '-v', '/home/led/temp/proGAT/temp/output/checkm2/hifiasm:/output', '-v', '/home/led/database/checkm2/CheckM2_database/uniref100.KO.1.dmnd:/checkm_data.dmnd:ro', 'staphb/checkm2:1.1.0', 'checkm2', 'predict', '--threads', '128', '--input', '/input.fasta', '--output-directory', '/output', '--database', '/checkm_data.dmnd'], returncode=0)

## 5.6 质粒预测-Plasmer

In [ ]:
plasmer_dir = os.path.join(output_dir, "plasmer")
os.makedirs(plasmer_dir, exist_ok=True)
docker_image_plasmer = "nekokoe/plasmer:23.04.20"

# 6. 物种鉴定-kraken2 

In [143]:
kraken2_dir = os.path.join(output_dir, "kraken2")
os.makedirs(kraken2_dir, exist_ok=True)
docker_image_kraken2 = "staphb/kraken2:2.17.1"

kraken2_db = os.path.join(database_path, "kraken2")

In [144]:
# flye
temp_dir = os.path.join(kraken2_dir, "flye")
os.makedirs(temp_dir, exist_ok=True)

input_file =os.path.join(flye_dir, "assembly.fasta")

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta:ro", 
    "-v", f"{temp_dir}:/output",
    "-v", f"{kraken2_db}:/kraken_db:ro", 
    docker_image_kraken2, 
    "kraken2", 
    "--db", "/kraken_db", 
    "--threads", "128", 
    "--report", "/output/kraken2_report.txt", 
    "--output", "/output/kraken2_output.txt", 
    "/input.fasta"
]

subprocess.run(cmd, check=True)

Loading database information... done.
5 sequences (5.75 Mbp) processed in 0.360s (0.8 Kseq/m, 959.54 Mbp/m).
  5 sequences classified (100.00%)
  0 sequences unclassified (0.00%)


CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/flye/assembly.fasta:/input.fasta:ro', '-v', '/home/led/temp/proGAT/temp/output/kraken2/flye:/output', '-v', '/home/led/database/kraken2:/kraken_db:ro', 'staphb/kraken2:2.17.1', 'kraken2', '--db', '/kraken_db', '--threads', '128', '--report', '/output/kraken2_report.txt', '--output', '/output/kraken2_output.txt', '/input.fasta'], returncode=0)

In [145]:
# hifiasm
temp_dir = os.path.join(kraken2_dir, "hifiasm")
os.makedirs(temp_dir, exist_ok=True)

input_file =os.path.join(hifiasm_dir, "assembly.fasta")

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta:ro", 
    "-v", f"{temp_dir}:/output",
    "-v", f"{kraken2_db}:/kraken_db:ro", 
    docker_image_kraken2, 
    "kraken2", 
    "--db", "/kraken_db", 
    "--threads", "128", 
    "--report", "/output/kraken2_report.txt", 
    "--output", "/output/kraken2_output.txt", 
    "/input.fasta"
]

subprocess.run(cmd, check=True)

Loading database information... done.
5 sequences (5.99 Mbp) processed in 0.360s (0.8 Kseq/m, 999.96 Mbp/m).
  5 sequences classified (100.00%)
  0 sequences unclassified (0.00%)


CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/hifiasm/assembly.fasta:/input.fasta:ro', '-v', '/home/led/temp/proGAT/temp/output/kraken2/hifiasm:/output', '-v', '/home/led/database/kraken2:/kraken_db:ro', 'staphb/kraken2:2.17.1', 'kraken2', '--db', '/kraken_db', '--threads', '128', '--report', '/output/kraken2_report.txt', '--output', '/output/kraken2_output.txt', '/input.fasta'], returncode=0)

# 7. CDS预测-Prokka

In [148]:
prokka_dir = os.path.join(output_dir, "prokka")
os.makedirs(prokka_dir, exist_ok=True)
docker_image_prokka = "staphb/prokka:1.15.6"

In [154]:
# flye
temp_dir = os.path.join(prokka_dir, "flye")
os.makedirs(temp_dir, exist_ok=True)

input_file =os.path.join(flye_dir, "assembly.fasta")

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta", 
    "-v", f"{temp_dir}:/output", 
    docker_image_prokka, 
    "prokka", 
    "--compliant", 
    "--outdir", "/output", 
    "--locustag", "contig", 
    "--prefix", "genomic",
    "--force", 
    "--cpus", "128", 
    "/input.fasta"
]

subprocess.run(cmd, check=True)

[08:55:25] This is prokka 1.15.6
[08:55:25] Written by Torsten Seemann <torsten.seemann@gmail.com>
[08:55:25] Homepage is https://github.com/tseemann/prokka
[08:55:25] Local time is Wed Sep 30 08:55:25 2026
[08:55:25] You are not telling me who you are!
[08:55:25] Operating system is linux
[08:55:25] You have BioPerl 1.7.8
[08:55:25] System has 192 cores.
[08:55:25] Will use maximum of 128 cores.
[08:55:25] Annotating as >>> Bacteria <<<
[08:55:25] Enabling options to ensure Genbank/ENA/DDJB submission compliance.
[08:55:25] Re-using existing --outdir /output
[08:55:25] Using filename prefix: genomic.XXX
[08:55:25] Setting HMMER_NCPU=1
[08:55:25] Writing log to: /output/genomic.log
[08:55:25] Command: /prokka-1.15.6/bin/prokka --compliant --outdir /output --locustag contig --prefix genomic --force --cpus 128 /input.fasta
[08:55:25] Looking for 'aragorn' - found /usr/bin/aragorn
[08:55:25] Determined aragorn version is v1.2 from 'ARAGORN v1.2.41 Dean Laslett'
[08:55:25] Looking for 'bar

CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/flye/assembly.fasta:/input.fasta', '-v', '/home/led/temp/proGAT/temp/output/prokka/flye:/output', 'staphb/prokka:1.15.6', 'prokka', '--compliant', '--outdir', '/output', '--locustag', 'contig', '--prefix', 'genomic', '--force', '--cpus', '128', '/input.fasta'], returncode=0)

In [155]:
# hifiasm
temp_dir = os.path.join(prokka_dir, "hifiasm")
os.makedirs(temp_dir, exist_ok=True)

input_file =os.path.join(hifiasm_dir, "assembly.fasta")

cmd = [
    "docker", "run", "--rm", 
    "-v", f"{input_file}:/input.fasta", 
    "-v", f"{temp_dir}:/output", 
    docker_image_prokka, 
    "prokka", 
    "--compliant", 
    "--outdir", "/output", 
    "--locustag", "contig", 
    "--prefix", "genomic",
    "--force", 
    "--cpus", "128", 
    "/input.fasta"
]

subprocess.run(cmd, check=True)

[08:57:00] This is prokka 1.15.6
[08:57:00] Written by Torsten Seemann <torsten.seemann@gmail.com>
[08:57:00] Homepage is https://github.com/tseemann/prokka
[08:57:00] Local time is Wed Sep 30 08:57:00 2026
[08:57:00] You are not telling me who you are!
[08:57:00] Operating system is linux
[08:57:00] You have BioPerl 1.7.8
[08:57:00] System has 192 cores.
[08:57:00] Will use maximum of 128 cores.
[08:57:00] Annotating as >>> Bacteria <<<
[08:57:00] Enabling options to ensure Genbank/ENA/DDJB submission compliance.
[08:57:00] Re-using existing --outdir /output
[08:57:00] Using filename prefix: genomic.XXX
[08:57:00] Setting HMMER_NCPU=1
[08:57:00] Writing log to: /output/genomic.log
[08:57:00] Command: /prokka-1.15.6/bin/prokka --compliant --outdir /output --locustag contig --prefix genomic --force --cpus 128 /input.fasta
[08:57:00] Looking for 'aragorn' - found /usr/bin/aragorn
[08:57:00] Determined aragorn version is v1.2 from 'ARAGORN v1.2.41 Dean Laslett'
[08:57:00] Looking for 'bar

CompletedProcess(args=['docker', 'run', '--rm', '-v', '/home/led/temp/proGAT/temp/output/hifiasm/assembly.fasta:/input.fasta', '-v', '/home/led/temp/proGAT/temp/output/prokka/hifiasm:/output', 'staphb/prokka:1.15.6', 'prokka', '--compliant', '--outdir', '/output', '--locustag', 'contig', '--prefix', 'genomic', '--force', '--cpus', '128', '/input.fasta'], returncode=0)

# 8. 假基因预测-PseudoFinder

In [ ]:
pseudofinder_dir = os.path.join(output_dir, "pseudofinder")
os.makedirs(pseudofinder_dir, exist_ok=True)
docker_image_pseudofinder = "figueredobea/pseudofinder:1.1.0"